# Tablero de Calidad de Datos — Primas (solo lectura)

Este notebook es **solo de lectura**: ejecuta únicamente `SELECT` contra Redshift (nunca `CREATE`/`INSERT`/`DELETE`/`CALL`). No depende de que el cubo `co_sandbox_datos.kpi_cubo_mensual` exista ni esté poblado — todo se calcula al vuelo desde la vista fuente.

Qué hace:

1. Se conecta a Redshift.
2. **Detecta automáticamente** los últimos periodos contables cerrados en la fuente (`gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement`) — sin periodos quemados, a diferencia de los `sql/01..05` originales.
3. Calcula los 5 indicadores (Completitud, Exactitud, Unicidad, Validez, Disponibilidad) con `SELECT` directos sobre la fuente, usando las mismas reglas de negocio de los archivos originales. Única desviación deliberada: aquí Exactitud **sí** aplica el filtro compartido completo (incluye `current_record_flag = 1`) — ver hallazgo #1 en [`docs/Hallazgos y Estado del Proyecto.md`](../docs/Hallazgos%20y%20Estado%20del%20Proyecto.md).
4. Genera un **tablero HTML autocontenido** (sin dependencias externas) en `reports/tablero_calidad_primas.html` y lo abre en el navegador.

In [37]:
import json
import webbrowser
from datetime import datetime
from pathlib import Path

import pandas as pd
import redshift_connector

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)

## Conexion a Redshift
Ajusta host/puerto/base si hace falta; usuario y contraseña se piden de forma interactiva.

In [38]:
import getpass

In [39]:
# Credenciales fuera del repo: se cargan desde notebooks/credenciales_local.py,
# que está en .gitignore (así no se exponen y tampoco hay que digitarlas cada vez).
# Si el archivo no existe, se piden por consola sin quedar guardadas en ninguna parte.
try:
    from credenciales_local import (
        DEFAULT_HOST, DEFAULT_PORT, DEFAULT_DATABASE, DEFAULT_USER, DEFAULT_PASSWORD
    )
    print('Credenciales cargadas desde credenciales_local.py (archivo local, fuera de git).')
except ImportError:
    print('No se encontró credenciales_local.py — ingresa las credenciales:')
    DEFAULT_HOST = 'corshftanltc-dprogramp.hdicolombia.com.co'
    DEFAULT_PORT = '9519'
    DEFAULT_DATABASE = 'adp_dwh'
    DEFAULT_USER = input('Usuario Redshift: ')
    DEFAULT_PASSWORD = getpass.getpass('Contraseña Redshift: ')

Credenciales cargadas desde credenciales_local.py (archivo local, fuera de git).


In [40]:
host = DEFAULT_HOST
port = DEFAULT_PORT
database =  DEFAULT_DATABASE
user = DEFAULT_USER
password = DEFAULT_PASSWORD

In [41]:
conn = redshift_connector.connect(
    host=host,
    port=int(port),
    database=database,
    user=user,
    password=password,
)
cursor = conn.cursor()
print('Conectado (solo lectura).')


Conectado (solo lectura).


In [42]:
def run_query(sql):
    """Ejecuta un SELECT (solo lectura) y devuelve un DataFrame."""
    cursor.execute(sql)
    return cursor.fetch_dataframe()


## 1) Detección automática de periodos (sin fechas quemadas)

Se listan los periodos disponibles en la fuente y se toman los últimos `N_PERIODOS` **cerrados** (menores al mes actual, porque el mes en curso todavía está incompleto). Cuando cierre un nuevo mes, basta volver a ejecutar el notebook — no hay nada que editar. Si necesitas forzar periodos puntuales, usa `PERIODOS_MANUAL`.

In [43]:
N_PERIODOS = 6          # cuántos periodos cerrados incluir en el tablero
PERIODOS_MANUAL = None  # ej. [202601, 202602] para forzar periodos puntuales

df_periodos = run_query('''
select distinct accountable_period as periodo
from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
where current_record_flag = 1
order by 1;
''')

periodo_mes_actual = int(datetime.now().strftime('%Y%m'))
periodos_cerrados = sorted(int(p) for p in df_periodos['periodo'] if int(p) < periodo_mes_actual)

PERIODOS = sorted(int(p) for p in PERIODOS_MANUAL) if PERIODOS_MANUAL else periodos_cerrados[-N_PERIODOS:]
if not PERIODOS:
    raise ValueError('No se encontraron periodos cerrados en la fuente.')
periodos_sql = ', '.join(str(p) for p in PERIODOS)

print(f'Mes actual (excluido por no estar cerrado): {periodo_mes_actual}')
print(f'Periodos del tablero: {PERIODOS}')

Mes actual (excluido por no estar cerrado): 202607
Periodos del tablero: [202601, 202602, 202603, 202604, 202605, 202606]


## 2) Cálculo de los 5 indicadores (solo `SELECT`)

Cada bloque ejecuta **una sola consulta agregada** para todos los periodos elegidos y deja el resultado en formato largo (`tipo_indicador` + `nombre_campo` + `periodo_contable`), el mismo grano del cubo unificado.

- Completitud, Exactitud, Unicidad y Validez usan el **filtro base compartido** documentado en `CLAUDE.md`.
- Disponibilidad **no** lo usa a propósito (compara tabla base vs. vista completas para no esconder brechas de sincronización).

In [44]:
# Filtro base compartido (Completitud / Exactitud / Unicidad / Validez) — ver CLAUDE.md.
FILTRO_BASE = f'''accountable_period IN ({periodos_sql})
  AND current_record_flag = 1
  AND coverage_code <> 8888
  AND transaction_delta_billed_premium_amount <> 0
  AND (
        (source_system = 'CO_iaxis' AND receipt_type NOT IN ('unificado-total'))
        OR source_system = 'CO_as400'
      )'''


def a_formato_largo(df_ancho, tipo_indicador):
    """Convierte un DF con una columna por campo (conteo de filas malas) a formato largo."""
    df = df_ancho.melt(
        id_vars=['periodo_contable', 'total_registros'],
        var_name='nombre_campo', value_name='cantidad_mala',
    )
    df['periodo_contable'] = df['periodo_contable'].astype('int64')
    df['total_registros'] = df['total_registros'].astype('int64')
    df['cantidad_mala'] = df['cantidad_mala'].astype('int64')
    df['tipo_indicador'] = tipo_indicador
    df['porcentaje'] = (100.0 * (1 - df['cantidad_mala'] / df['total_registros'])).round(2)
    df['es_total'] = 0
    return df


# ---- COMPLETITUD: % de no-nulos por campo (19 obligatorios + 3 con excepciones por fuente)
CAMPOS_OBLIGATORIOS = [
    'source_system', 'accountable_period', 'coverage_code', 'branch_sk', 'product_code',
    'transaction_date_sk', 'policy_effective_date_sk', 'policy_expiration_date_sk',
    'inception_date_sk', 'transaction_type', 'transaction_effective_date_sk',
    'transaction_type_description', 'current_record_flag',
    'transaction_delta_billed_premium_amount', 'transaction_delta_commission_amount',
    'risk_number', 'policy_number', 'transaction_delta_billed_premium_amount_raw',
    'policy_transaction_movement_sk',
]
nulls_simples = ',\n    '.join(
    f'SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}' for c in CAMPOS_OBLIGATORIOS
)

sql_completitud = f'''
select
    accountable_period as periodo_contable,
    count(*) as total_registros,
    {nulls_simples},
    -- Campos con excepciones por fuente (CO_as400 no los puebla):
    SUM(CASE WHEN sseguro IS NULL AND source_system <> 'CO_as400' THEN 1 ELSE 0 END) AS sseguro,
    SUM(CASE WHEN receipt_type IS NULL AND source_system <> 'CO_as400' THEN 1 ELSE 0 END) AS receipt_type,
    SUM(CASE
            WHEN (receipt_number IS NULL AND source_system <> 'CO_as400' AND receipt_type <> 'not-unificado')
              OR (receipt_type = 'not-unificado' AND receipt_number IS NOT NULL)
            THEN 1 ELSE 0
        END) AS receipt_number
from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
where {FILTRO_BASE}
group by 1
order by 1;
'''

df_completitud = a_formato_largo(run_query(sql_completitud), 'completitud')
print(f'{len(df_completitud)} filas (campo x periodo). Peores campos del último periodo:')
df_completitud[df_completitud['periodo_contable'] == PERIODOS[-1]].sort_values('porcentaje').head(10)

132 filas (campo x periodo). Peores campos del último periodo:


,periodo_contable,total_registros,nombre_campo,cantidad_mala,tipo_indicador,porcentaje,es_total
5,202606,4672974,source_system,0,completitud,100.0,0
11,202606,4672974,accountable_period,0,completitud,100.0,0
17,202606,4672974,coverage_code,0,completitud,100.0,0
23,202606,4672974,branch_sk,0,completitud,100.0,0
29,202606,4672974,product_code,1,completitud,100.0,0
35,202606,4672974,transaction_date_sk,0,completitud,100.0,0
41,202606,4672974,policy_effective_date_sk,0,completitud,100.0,0
47,202606,4672974,policy_expiration_date_sk,0,completitud,100.0,0
53,202606,4672974,inception_date_sk,0,completitud,100.0,0
59,202606,4672974,transaction_type,0,completitud,100.0,0


In [45]:
# ---- EXACTITUD: valores dentro del dominio permitido (3 reglas implementadas).
# Nota deliberada: aquí SÍ se aplica el filtro compartido completo (incluye
# current_record_flag = 1), a diferencia del sql/03 original — hallazgo #1 del
# doc de hallazgos. Por eso las cifras pueden diferir levemente del original.
# Las reglas contra tabla ODS (transaction_delta_*) siguen sin implementar (hallazgo #2).
sql_exactitud = f'''
select
    accountable_period as periodo_contable,
    count(*) as total_registros,
    SUM(CASE WHEN source_system IS NULL OR source_system NOT IN ('CO_iaxis', 'CO_as400') THEN 1 ELSE 0 END) AS source_system,
    SUM(CASE WHEN current_record_flag IS NULL OR current_record_flag NOT IN (0, 1) THEN 1 ELSE 0 END) AS current_record_flag,
    SUM(CASE WHEN receipt_type IS NULL OR receipt_type NOT IN ('not-unificado', 'unificado-detail', 'unificado-total') THEN 1 ELSE 0 END) AS receipt_type
from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
where {FILTRO_BASE}
group by 1
order by 1;
'''

df_exactitud = a_formato_largo(run_query(sql_exactitud), 'exactitud')
df_exactitud

,periodo_contable,total_registros,nombre_campo,cantidad_mala,tipo_indicador,porcentaje,es_total
0,202601,4930056,source_system,0,exactitud,100.00,0
1,202602,4547533,source_system,0,exactitud,100.00,0
2,202603,4549776,source_system,0,exactitud,100.00,0
3,202604,4639530,source_system,0,exactitud,100.00,0
4,202605,4738110,source_system,0,exactitud,100.00,0
5,202606,4672974,source_system,0,exactitud,100.00,0
6,202601,4930056,current_record_flag,0,exactitud,100.00,0
7,202602,4547533,current_record_flag,0,exactitud,100.00,0
8,202603,4549776,current_record_flag,0,exactitud,100.00,0
9,202604,4639530,current_record_flag,0,exactitud,100.00,0


In [46]:
# ---- UNICIDAD: duplicados de policy_transaction_movement_sk dentro del universo filtrado.
sql_unicidad = f'''
with base as (
    select accountable_period as periodo_contable, policy_transaction_movement_sk
    from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
    where {FILTRO_BASE}
),
duplicados as (
    select periodo_contable, policy_transaction_movement_sk
    from base
    group by 1, 2
    having count(*) > 1
)
select
    b.periodo_contable,
    count(*) as total_registros,
    SUM(CASE WHEN d.policy_transaction_movement_sk IS NOT NULL THEN 1 ELSE 0 END) as cantidad_mala
from base b
left join duplicados d
  on d.periodo_contable = b.periodo_contable
 and d.policy_transaction_movement_sk = b.policy_transaction_movement_sk
group by 1
order by 1;
'''

df_unicidad = run_query(sql_unicidad)
df_unicidad['periodo_contable'] = df_unicidad['periodo_contable'].astype('int64')
df_unicidad['total_registros'] = df_unicidad['total_registros'].astype('int64')
df_unicidad['cantidad_mala'] = df_unicidad['cantidad_mala'].astype('int64')
df_unicidad['tipo_indicador'] = 'unicidad'
df_unicidad['nombre_campo'] = 'policy_transaction_movement_sk'
df_unicidad['porcentaje'] = (
    100.0 * (df_unicidad['total_registros'] - df_unicidad['cantidad_mala']) / df_unicidad['total_registros']
).round(2)
df_unicidad['es_total'] = 1
df_unicidad

,periodo_contable,total_registros,cantidad_mala,tipo_indicador,nombre_campo,porcentaje,es_total
0,202601,4930056,0,unicidad,policy_transaction_movement_sk,100.0,1
1,202602,4547533,0,unicidad,policy_transaction_movement_sk,100.0,1
2,202603,4549776,0,unicidad,policy_transaction_movement_sk,100.0,1
3,202604,4639530,0,unicidad,policy_transaction_movement_sk,100.0,1
4,202605,4738110,0,unicidad,policy_transaction_movement_sk,100.0,1
5,202606,4672974,0,unicidad,policy_transaction_movement_sk,100.0,1


In [47]:
# ---- VALIDEZ: formato/dominio por campo (12 reglas).
# movement_type está excluido a propósito (comentado también en el sql/05 original).
REGLAS_VALIDEZ = {
    'source_system': "source_system IS NULL OR source_system = 'Unknown'",
    'coverage_code': "coverage_code IS NULL OR coverage_code = 'Unknown'",
    'product_code': (
        "product_code IS NULL OR product_code = 'Unknown' OR LENGTH(product_code) > 6 "
        "OR product_code LIKE '% %' OR product_code ~ '[^A-Za-z0-9]' "
        "OR (source_system = 'CO_iaxis' AND product_code ~ '[A-Za-z]')"
    ),
    'transaction_date_sk': "transaction_date_sk IS NULL OR transaction_date_sk::VARCHAR !~ '^[0-9]{8}$'",
    'transaction_type': "transaction_type IS NULL OR transaction_type = 'Unknown' OR LENGTH(transaction_type) > 2",
    'current_record_flag': "current_record_flag IS NULL OR current_record_flag NOT IN (0, 1)",
    'risk_number': "risk_number IS NULL OR risk_number = 'Unknown'",
    'policy_number': "policy_number IS NULL OR policy_number = 'Unknown'",
    'accountable_period': "accountable_period IS NULL OR LENGTH(accountable_period::VARCHAR) <> 6",
    'policy_effective_date_sk': "policy_effective_date_sk IS NULL OR LENGTH(policy_effective_date_sk::VARCHAR) <> 8",
    'inception_date_sk': "inception_date_sk IS NULL OR LENGTH(inception_date_sk::VARCHAR) <> 8",
    'transaction_effective_date_sk': "transaction_effective_date_sk IS NULL OR LENGTH(transaction_effective_date_sk::VARCHAR) <> 8",
}

casos_validez = ',\n    '.join(
    f'SUM(CASE WHEN {regla} THEN 1 ELSE 0 END) AS {campo}'
    for campo, regla in REGLAS_VALIDEZ.items()
)

sql_validez = f'''
select
    accountable_period as periodo_contable,
    count(*) as total_registros,
    {casos_validez}
from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
where {FILTRO_BASE}
group by 1
order by 1;
'''

df_validez = a_formato_largo(run_query(sql_validez), 'validez')
print(f'{len(df_validez)} filas (campo x periodo). Peores campos del último periodo:')
df_validez[df_validez['periodo_contable'] == PERIODOS[-1]].sort_values('porcentaje').head(10)

72 filas (campo x periodo). Peores campos del último periodo:


,periodo_contable,total_registros,nombre_campo,cantidad_mala,tipo_indicador,porcentaje,es_total
11,202606,4672974,coverage_code,368,validez,99.99,0
5,202606,4672974,source_system,0,validez,100.00,0
17,202606,4672974,product_code,1,validez,100.00,0
23,202606,4672974,transaction_date_sk,0,validez,100.00,0
29,202606,4672974,transaction_type,0,validez,100.00,0
35,202606,4672974,current_record_flag,0,validez,100.00,0
41,202606,4672974,risk_number,0,validez,100.00,0
47,202606,4672974,policy_number,0,validez,100.00,0
53,202606,4672974,accountable_period,0,validez,100.00,0
59,202606,4672974,policy_effective_date_sk,0,validez,100.00,0


In [48]:
# ---- Verificación de acceso a la capa base (gde_adp_dwh) ----
# disponibilidad_sync compara tabla base vs. vista y requiere SELECT sobre
# gde_adp_dwh.fact_policy_transaction_movement. Esta sonda barata (limit 1) detecta
# si hay permiso ANTES de lanzar la query pesada; si no lo hay, ese chequeo se omite.
TIENE_ACCESO_DWH = False
try:
    run_query('select 1 from gde_adp_dwh.fact_policy_transaction_movement limit 1')
    TIENE_ACCESO_DWH = True
    print('OK: hay acceso a gde_adp_dwh.fact_policy_transaction_movement — se calculará disponibilidad_sync.')
except Exception as e:
    conn.rollback()  # la transacción queda abortada tras el error; sin esto fallan las celdas siguientes
    print(f'SIN ACCESO a gde_adp_dwh.fact_policy_transaction_movement ({type(e).__name__}).')
    print('disponibilidad_sync se omitirá; el resto de indicadores no se afecta.')

SIN ACCESO a gde_adp_dwh.fact_policy_transaction_movement (ProgrammingError).
disponibilidad_sync se omitirá; el resto de indicadores no se afecta.


In [49]:
# ---- DISPONIBILIDAD: dos chequeos NO comparables entre sí (ver CLAUDE.md y sql/06).
# A propósito SIN el filtro base compartido: compara tabla base vs. vista completas.

# (a) disponibilidad_sync: conciliación tabla base vs. vista (desfase de refresco).
#     ~98% es esperado/normal, no una falla.
#     OJO: requiere SELECT sobre la tabla base gde_adp_dwh.fact_policy_transaction_movement.
#     Si el usuario no tiene ese permiso (error 42501), el chequeo se omite con una
#     advertencia y el tablero lo muestra como "sin datos" — el resto sigue funcionando.
sql_disponibilidad_sync = f'''
with base_dwh as (
    select cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) as periodo_contable,
           policy_transaction_movement_sk
    from gde_adp_dwh.fact_policy_transaction_movement
    where cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) in ({periodos_sql})
),
base_vista as (
    select cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) as periodo_contable,
           policy_transaction_movement_sk
    from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
    where cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) in ({periodos_sql})
),
totales as (
    select periodo_contable, count(*) as total_registros
    from base_dwh
    group by 1
),
faltantes as (
    select d.periodo_contable, count(*) as cantidad_mala
    from base_dwh d
    left join base_vista v
      on v.periodo_contable = d.periodo_contable
     and v.policy_transaction_movement_sk = d.policy_transaction_movement_sk
    where v.policy_transaction_movement_sk is null
    group by 1
)
select t.periodo_contable, t.total_registros, coalesce(f.cantidad_mala, 0) as cantidad_mala
from totales t
left join faltantes f on f.periodo_contable = t.periodo_contable
order by 1;
'''

# (b) disponibilidad_regla4: % de ramos (product_code) con datos TODOS los días 1-15 del mes.
#     Solo usa la vista, así que no tiene el problema de permisos.
#     Porcentajes bajos son comunes (basta 1 día faltante para castigar el ramo).
sql_disponibilidad_regla4 = f'''
with base as (
    select cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) as periodo_contable,
           product_code,
           cast(substring(cast(transaction_date_sk as varchar), 7, 2) as integer) as dia
    from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
    where cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) in ({periodos_sql})
),
ramos as (
    select periodo_contable, product_code,
           count(distinct case when dia between 1 and 15 then dia end) as dias_presentes
    from base
    group by 1, 2
)
select
    periodo_contable,
    count(*) as total_registros,                                        -- total de ramos
    SUM(CASE WHEN dias_presentes = 15 THEN 0 ELSE 1 END) as cantidad_mala  -- ramos con algún día 1-15 faltante
from ramos
group by 1
order by 1;
'''

COLUMNAS_DISP = ['periodo_contable', 'total_registros', 'cantidad_mala',
                 'tipo_indicador', 'nombre_campo', 'porcentaje', 'es_total']
TIENE_ACCESO_DWH = globals().get('TIENE_ACCESO_DWH', False)  # False si no corrió la celda de verificación

partes = []
for campo, sql in [('disponibilidad_sync', sql_disponibilidad_sync),
                   ('disponibilidad_regla4', sql_disponibilidad_regla4)]:
    if campo == 'disponibilidad_sync' and not TIENE_ACCESO_DWH:
        print('AVISO: disponibilidad_sync omitido — sin permiso SELECT sobre gde_adp_dwh (ver celda anterior).')
        continue
    try:
        df = run_query(sql)
    except Exception as e:
        conn.rollback()  # la transacción queda abortada tras el error; sin esto fallan las celdas siguientes
        print(f'AVISO: {campo} omitido — {type(e).__name__}: {str(e)[:200]}')
        if campo == 'disponibilidad_sync':
            print('  (Este chequeo requiere SELECT sobre gde_adp_dwh.fact_policy_transaction_movement.')
            print('   Sin ese permiso no se puede comparar tabla base vs. vista; pídelo si necesitas el sync.)')
        continue
    df['periodo_contable'] = df['periodo_contable'].astype('int64')
    df['total_registros'] = df['total_registros'].astype('int64')
    df['cantidad_mala'] = df['cantidad_mala'].astype('int64')
    df['tipo_indicador'] = 'disponibilidad'
    df['nombre_campo'] = campo
    df['porcentaje'] = (
        100.0 * (df['total_registros'] - df['cantidad_mala']) / df['total_registros']
    ).round(2)
    df['es_total'] = 1
    partes.append(df[COLUMNAS_DISP])

if partes:
    df_disponibilidad = pd.concat(partes, ignore_index=True)
else:
    df_disponibilidad = pd.DataFrame(columns=COLUMNAS_DISP)
df_disponibilidad

AVISO: disponibilidad_sync omitido — sin permiso SELECT sobre gde_adp_dwh (ver celda anterior).


,periodo_contable,total_registros,cantidad_mala,tipo_indicador,nombre_campo,porcentaje,es_total
0,202601,85,82,disponibilidad,disponibilidad_regla4,3.53,1
1,202602,84,77,disponibilidad,disponibilidad_regla4,8.33,1
2,202603,82,76,disponibilidad,disponibilidad_regla4,7.32,1
3,202604,85,80,disponibilidad,disponibilidad_regla4,5.88,1
4,202605,78,75,disponibilidad,disponibilidad_regla4,3.85,1
5,202606,86,78,disponibilidad,disponibilidad_regla4,9.30,1


## 3) Consolidar el cubo en formato largo

Se unen los 5 indicadores en un solo DataFrame con el mismo grano del cubo unificado (`periodo_contable` + `tipo_indicador` + `nombre_campo`) y se agregan las filas de total por indicador (`TOTAL_PERIODO` / `TOTAL`, promedio simple de los porcentajes por campo — igual que el cubo).

In [50]:
COLUMNAS_CUBO = ['periodo_contable', 'tipo_indicador', 'nombre_campo',
                 'total_registros', 'cantidad_mala', 'porcentaje', 'es_total']

df_cubo = pd.concat(
    [df_completitud, df_exactitud, df_unicidad, df_validez, df_disponibilidad],
    ignore_index=True,
)[COLUMNAS_CUBO]

# Filas de total por indicador (promedio simple de los porcentajes por campo, como el cubo)
totales = []
for tipo, nombre in [('completitud', 'TOTAL_PERIODO'), ('exactitud', 'TOTAL'), ('validez', 'TOTAL_PERIODO')]:
    t = (df_cubo[df_cubo['tipo_indicador'] == tipo]
         .groupby('periodo_contable')
         .agg(total_registros=('total_registros', 'max'),
              cantidad_mala=('cantidad_mala', 'sum'),
              porcentaje=('porcentaje', 'mean'))
         .reset_index())
    t['porcentaje'] = t['porcentaje'].round(2)
    t['tipo_indicador'] = tipo
    t['nombre_campo'] = nombre
    t['es_total'] = 1
    totales.append(t[COLUMNAS_CUBO])

df_cubo = pd.concat([df_cubo] + totales, ignore_index=True).sort_values(
    ['periodo_contable', 'tipo_indicador', 'es_total', 'nombre_campo']
).reset_index(drop=True)

# Vista rápida: porcentaje total por indicador y periodo
df_cubo[df_cubo['es_total'] == 1].pivot_table(
    index=['tipo_indicador', 'nombre_campo'],
    columns='periodo_contable',
    values='porcentaje',
)

,periodo_contable,202601,202602,202603,202604,202605,202606
tipo_indicador,nombre_campo,,,,,,
completitud,TOTAL_PERIODO,100.00,100.00,100.00,100.00,100.00,100.00
disponibilidad,disponibilidad_regla4,3.53,8.33,7.32,5.88,3.85,9.30
exactitud,TOTAL,97.83,97.69,97.72,97.73,97.82,97.74
unicidad,policy_transaction_movement_sk,100.00,100.00,100.00,100.00,100.00,100.00
validez,TOTAL_PERIODO,100.00,100.00,100.00,100.00,100.00,100.00


## 4) Generar el tablero HTML de calidad

Se arma un archivo HTML **autocontenido** (sin dependencias externas, funciona sin internet) a partir de `df_cubo`, se guarda en `reports/tablero_calidad_primas.html` y se abre en el navegador. Incluye:

- Selector de periodo (todos los periodos calculados).
- Una tarjeta por indicador con el porcentaje total, la variación vs. el periodo anterior, el estado y la tendencia.
- Una tabla por indicador con el detalle por campo (registros, conteo de malos, porcentaje y estado).

Los umbrales de estado (`BUENO`/`ALERTA`/`GRAVE`/`CRÍTICO`) se ajustan en la celda siguiente. Disponibilidad tiene umbrales propios porque sus dos chequeos no se interpretan igual que los demás (ver notas en el propio tablero y en `CLAUDE.md`).

In [51]:
# Umbrales de estado por porcentaje (ajustables).
# Regla: >= BUENO -> BUENO; >= ALERTA -> ALERTA; >= GRAVE -> GRAVE; por debajo -> CRÍTICO.
#
# Disponibilidad tiene umbrales propios (ver CLAUDE.md):
#  - disponibilidad_sync: ~98% es esperado/normal (desfase de refresco vista vs. tabla), no una falla.
#  - disponibilidad_regla4: suele dar porcentajes bajos (basta 1 día faltante 1-15 para castigar el ramo).
UMBRALES = {
    'default':               {'BUENO': 99.0, 'ALERTA': 97.0, 'GRAVE': 90.0},
    'disponibilidad_sync':   {'BUENO': 97.0, 'ALERTA': 90.0, 'GRAVE': 80.0},
    'disponibilidad_regla4': {'BUENO': 80.0, 'ALERTA': 60.0, 'GRAVE': 40.0},
}

In [52]:
HTML_TEMPLATE = r'''<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Calidad de Datos — Primas</title>
<style>
  :root{
    --surface:#fcfcfb; --page:#f9f9f7;
    --ink:#0b0b0b; --ink-2:#52514e; --muted:#898781;
    --grid:#e1e0d9; --border:rgba(11,11,11,.10);
    --accent:#2a78d6; --track:#cde2fb; --spark:#c3c2b7;
    --good:#0ca30c; --warn:#fab219; --serious:#ec835a; --critical:#d03b3b;
    --delta-up:#006300; --delta-down:#d03b3b;
  }
  @media (prefers-color-scheme: dark){
    :root{
      --surface:#1a1a19; --page:#0d0d0d;
      --ink:#ffffff; --ink-2:#c3c2b7;
      --grid:#2c2c2a; --border:rgba(255,255,255,.10);
      --accent:#3987e5; --track:#0d366b; --spark:#52514e;
      --delta-up:#0ca30c;
    }
  }
  *{box-sizing:border-box}
  body{margin:0;background:var(--page);color:var(--ink);
       font:14px/1.45 system-ui,-apple-system,"Segoe UI",sans-serif}
  .wrap{max-width:1180px;margin:0 auto;padding:28px 24px 56px}
  h1{font-size:20px;font-weight:650;margin:0}
  .sub{color:var(--ink-2);font-size:13px;margin:4px 0 0}
  .filtros{display:flex;align-items:center;gap:12px;margin:22px 0 14px;flex-wrap:wrap}
  .filtros label{font-size:13px;color:var(--ink-2)}
  select{font:inherit;color:var(--ink);background:var(--surface);
         border:1px solid var(--border);border-radius:8px;padding:6px 10px}
  .tiles{display:grid;grid-template-columns:repeat(auto-fit,minmax(178px,1fr));gap:12px;margin-bottom:24px}
  .tile{background:var(--surface);border:1px solid var(--border);border-radius:10px;padding:14px 16px}
  .tile .lbl{font-size:12.5px;color:var(--ink-2)}
  .tile .val{font-size:29px;font-weight:600;letter-spacing:-.01em;margin:2px 0 1px}
  .tile .val small{font-size:15px;font-weight:500;color:var(--ink-2)}
  .tile .delta{font-size:12px;color:var(--muted);margin-bottom:8px;min-height:17px}
  .tile .delta .up{color:var(--delta-up);font-weight:600}
  .tile .delta .down{color:var(--delta-down);font-weight:600}
  .chip{display:inline-flex;align-items:center;gap:5px;font-size:11.5px;font-weight:600;
        padding:2px 9px;border-radius:999px;
        background:color-mix(in srgb, var(--c) 14%, transparent)}
  .chip .ico{color:var(--c);font-size:11px}
  section{background:var(--surface);border:1px solid var(--border);border-radius:10px;
          padding:18px 20px;margin-bottom:14px}
  section h2{font-size:15px;font-weight:650;margin:0 0 2px}
  section .nota{font-size:12.5px;color:var(--ink-2);margin:0 0 12px}
  .twrap{overflow-x:auto}
  table{border-collapse:collapse;width:100%;font-size:13px}
  th{color:var(--muted);font-weight:500;text-align:left;font-size:12px;
     padding:6px 10px;border-bottom:1px solid var(--grid);white-space:nowrap}
  th.num{text-align:right}
  td{padding:7px 10px;border-bottom:1px solid var(--grid)}
  tr:last-child td{border-bottom:none}
  td.num{text-align:right;font-variant-numeric:tabular-nums;white-space:nowrap}
  td.campo{font-family:ui-monospace,Consolas,monospace;font-size:12.5px}
  .track{height:8px;background:var(--track);border-radius:4px;min-width:110px}
  .fill{height:100%;background:var(--accent);border-radius:0 4px 4px 0}
  footer{color:var(--muted);font-size:12px;margin-top:20px}
  footer ul{margin:6px 0 0;padding-left:18px}
</style>
</head>
<body>
<div class="wrap">
  <h1>Calidad de Datos — Primas</h1>
  <p class="sub" id="sub"></p>
  <div class="filtros">
    <label for="periodo">Periodo contable</label>
    <select id="periodo"></select>
    <span id="leyenda" style="display:inline-flex;gap:8px;flex-wrap:wrap"></span>
  </div>
  <div class="tiles" id="tiles"></div>
  <div id="secciones"></div>
  <footer id="notas"></footer>
</div>
<script>
const DATA = __DATA__;
const fmtInt = new Intl.NumberFormat('es-CO').format;
const PERIODOS = DATA.periodos;

const ESTADOS = {
  'BUENO':   {c:'var(--good)',     i:'✓'},
  'ALERTA':  {c:'var(--warn)',     i:'▲'},
  'GRAVE':   {c:'var(--serious)',  i:'●'},
  'CRÍTICO': {c:'var(--critical)', i:'✕'},
};

const INDICADORES = [
  {tipo:'completitud', total:'TOTAL_PERIODO', label:'Completitud', malos:'Nulos',
   nota:'% de registros no nulos por campo obligatorio (el total es el promedio simple de los campos).'},
  {tipo:'exactitud', total:'TOTAL', label:'Exactitud', malos:'Inexactos',
   nota:'Valores dentro del dominio permitido. Aplica el filtro compartido completo (incluye current_record_flag = 1), a diferencia del SQL original de Exactitud — desviación deliberada documentada (hallazgo #1).'},
  {tipo:'unicidad', total:'policy_transaction_movement_sk', label:'Unicidad', malos:'Duplicados',
   nota:'Registros sin duplicado de policy_transaction_movement_sk dentro del universo filtrado.'},
  {tipo:'validez', total:'TOTAL_PERIODO', label:'Validez', malos:'Inválidos',
   nota:'Formato/dominio por campo (longitudes, formas YYYYMMDD/YYYYMM, caracteres permitidos). movement_type excluido a propósito.'},
  {tipo:'disponibilidad', total:'disponibilidad_sync', umbral:'disponibilidad_sync',
   label:'Disp. sincronización', malos:'Faltantes',
   nota:'disponibilidad_sync: tabla base vs. vista (desfase de refresco de la vista materializada); ~98% es esperado y normal, no una falla.'},
  {tipo:'disponibilidad', total:'disponibilidad_regla4', umbral:'disponibilidad_regla4',
   label:'Disp. regla 4 (días 1–15)', malos:'Ramos con faltantes',
   nota:'disponibilidad_regla4: % de ramos (product_code) con datos todos los días 1–15 del mes; porcentajes bajos son comunes (basta 1 día faltante para castigar el ramo).'},
];

function estado(clave, pct){
  const u = DATA.umbrales[clave] || DATA.umbrales['default'];
  if (pct >= u.BUENO) return 'BUENO';
  if (pct >= u.ALERTA) return 'ALERTA';
  if (pct >= u.GRAVE) return 'GRAVE';
  return 'CRÍTICO';
}
function chip(e){
  const s = ESTADOS[e];
  return '<span class="chip" style="--c:'+s.c+'"><span class="ico">'+s.i+'</span>'+e+'</span>';
}
function filaTotal(ind, periodo){
  return DATA.filas.find(f => f.tipo_indicador === ind.tipo
    && f.nombre_campo === ind.total && f.periodo_contable === periodo);
}
function sparkline(vals, selIdx){
  const w = 124, h = 30, pad = 5;
  const pts = vals.map((v, i) => ({v, i})).filter(d => d.v != null);
  if (!pts.length) return '';
  const min = Math.min(...pts.map(d => d.v)), max = Math.max(...pts.map(d => d.v));
  const span = (max - min) || 1;
  const x = i => PERIODOS.length > 1 ? pad + (w - 2*pad) * i / (PERIODOS.length - 1) : w / 2;
  const y = v => h - pad - (h - 2*pad) * (v - min) / span;
  const sel = pts.find(d => d.i === selIdx) || pts[pts.length - 1];
  let svg = '<svg width="'+w+'" height="'+h+'" role="img" aria-label="tendencia">';
  if (pts.length > 1){
    const poly = pts.map(d => x(d.i).toFixed(1)+','+y(d.v).toFixed(1)).join(' ');
    svg += '<polyline points="'+poly+'" fill="none" stroke="var(--spark)" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"/>';
  }
  svg += '<circle cx="'+x(sel.i).toFixed(1)+'" cy="'+y(sel.v).toFixed(1)+'" r="4" fill="var(--accent)" stroke="var(--surface)" stroke-width="2">'
       + '<title>'+PERIODOS[sel.i]+': '+sel.v.toFixed(2)+' %</title></circle></svg>';
  return svg;
}
function barra(p){
  const w = Math.max(0, Math.min(100, p));
  return '<div class="track"><div class="fill" style="width:'+w+'%"></div></div>';
}
function tr(campo, f, clave){
  const mala = f.cantidad_mala == null ? '—' : fmtInt(f.cantidad_mala);
  return '<tr><td class="campo">'+campo+'</td>'
    + '<td class="num">'+fmtInt(f.total_registros)+'</td>'
    + '<td class="num">'+mala+'</td>'
    + '<td class="num">'+f.porcentaje.toFixed(2)+'</td>'
    + '<td>'+barra(f.porcentaje)+'</td>'
    + '<td>'+chip(estado(clave, f.porcentaje))+'</td></tr>';
}
function tablaHtml(malosLbl, rowsHtml){
  return '<div class="twrap"><table><thead><tr>'
    + '<th>Campo</th><th class="num">Registros</th><th class="num">'+malosLbl+'</th>'
    + '<th class="num">%</th><th style="width:150px"></th><th>Estado</th>'
    + '</tr></thead><tbody>'+rowsHtml+'</tbody></table></div>';
}
function seccionIndicador(ind, periodo){
  let filas = DATA.filas.filter(f => f.tipo_indicador === ind.tipo
    && f.periodo_contable === periodo && !f.es_total);
  if (!filas.length){
    filas = DATA.filas.filter(f => f.tipo_indicador === ind.tipo && f.periodo_contable === periodo);
  }
  filas = filas.slice().sort((a, b) => a.porcentaje - b.porcentaje);
  const rows = filas.map(f => tr(f.nombre_campo, f, 'default')).join('');
  return '<section><h2>'+ind.label+'</h2><p class="nota">'+ind.nota+'</p>'
    + (filas.length ? tablaHtml(ind.malos, rows) : '<p class="nota">Sin datos para este periodo.</p>')
    + '</section>';
}
function seccionDisponibilidad(periodo){
  const inds = INDICADORES.filter(i => i.umbral);
  const rows = inds.map(ind => {
    const f = filaTotal(ind, periodo);
    return f ? tr(ind.total, f, ind.umbral) : '';
  }).join('');
  let aviso = '';
  if (!DATA.filas.some(f => f.nombre_campo === 'disponibilidad_sync')){
    aviso = '<p class="nota">⚠ disponibilidad_sync <strong>omitido</strong>: el usuario que generó el reporte no tiene '
      + 'SELECT sobre la tabla base gde_adp_dwh.fact_policy_transaction_movement (requerido para comparar tabla vs. vista).</p>';
  }
  return '<section><h2>Disponibilidad</h2>'
    + '<p class="nota">Dos chequeos <strong>no comparables entre sí</strong>. '
    + inds.map(i => i.nota).join(' ') + '</p>' + aviso
    + (rows ? tablaHtml('Faltantes', rows) : '<p class="nota">Sin datos para este periodo.</p>')
    + '</section>';
}
function render(){
  const periodo = parseInt(document.getElementById('periodo').value, 10);
  const selIdx = PERIODOS.indexOf(periodo);

  document.getElementById('tiles').innerHTML = INDICADORES.map(ind => {
    const fila = filaTotal(ind, periodo);
    const serie = PERIODOS.map(p => { const f = filaTotal(ind, p); return f ? f.porcentaje : null; });
    const clave = ind.umbral || 'default';
    let deltaHtml = '&nbsp;';
    if (fila && selIdx > 0 && serie[selIdx - 1] != null){
      const d = fila.porcentaje - serie[selIdx - 1];
      const cls = d >= 0 ? 'up' : 'down';
      deltaHtml = '<span class="'+cls+'">'+(d >= 0 ? '+' : '')+d.toFixed(2)+' pp</span> vs '+PERIODOS[selIdx - 1];
    }
    return '<div class="tile"><div class="lbl">'+ind.label+'</div>'
      + '<div class="val">'+(fila ? fila.porcentaje.toFixed(2) : '—')+'<small> %</small></div>'
      + '<div class="delta">'+deltaHtml+'</div>'
      + (fila ? chip(estado(clave, fila.porcentaje)) : '<span class="chip" style="--c:var(--muted)">sin datos</span>')
      + '<div style="margin-top:8px">'+sparkline(serie, selIdx)+'</div></div>';
  }).join('');

  document.getElementById('secciones').innerHTML =
    INDICADORES.filter(ind => !ind.umbral).map(ind => seccionIndicador(ind, periodo)).join('')
    + seccionDisponibilidad(periodo);
}

const sel = document.getElementById('periodo');
sel.innerHTML = PERIODOS.slice().reverse().map(p => '<option value="'+p+'">'+p+'</option>').join('');
sel.addEventListener('change', render);

document.getElementById('sub').textContent =
  'Fuente: ' + DATA.fuente + ' · Generado: ' + DATA.generado + ' · Solo lectura (SELECT directo, sin cubo persistido)';
document.getElementById('leyenda').innerHTML = Object.keys(ESTADOS).map(chip).join('');

const u = DATA.umbrales;
function umbTxt(k){
  const t = u[k];
  return 'BUENO ≥ '+t.BUENO+' · ALERTA ≥ '+t.ALERTA+' · GRAVE ≥ '+t.GRAVE+' · CRÍTICO < '+t.GRAVE;
}
document.getElementById('notas').innerHTML = '<strong>Notas</strong><ul>'
  + '<li>Umbrales generales: '+umbTxt('default')+' (ajustables en el notebook).</li>'
  + '<li>Disponibilidad sincronización: '+umbTxt('disponibilidad_sync')+'.</li>'
  + '<li>Disponibilidad regla 4: '+umbTxt('disponibilidad_regla4')+'.</li>'
  + '<li>Universo: filtro base compartido de CLAUDE.md (current_record_flag = 1, coverage_code ≠ 8888, prima ≠ 0, iaxis sin unificado-total + as400); Disponibilidad no lo aplica a propósito.</li>'
  + '</ul>';

render();
</script>
</body>
</html>'''

payload = {
    'generado': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'fuente': 'gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement',
    'periodos': [int(p) for p in PERIODOS],
    'umbrales': UMBRALES,
    'filas': json.loads(df_cubo.to_json(orient='records')),  # NaN -> null
}

html = HTML_TEMPLATE.replace('__DATA__', json.dumps(payload, ensure_ascii=False))

repo_raiz = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ruta_html = repo_raiz / 'reports' / 'tablero_calidad_primas.html'
ruta_html.parent.mkdir(exist_ok=True)
ruta_html.write_text(html, encoding='utf-8')
print(f'Tablero generado: {ruta_html}')
webbrowser.open(ruta_html.resolve().as_uri())

Tablero generado: c:\Users\Wilson.Jerez\OneDrive - HDI Seguros\Documentos\hdi-primas-data-quality\reports\tablero_calidad_primas.html


True

## Cerrar conexión

In [53]:
cursor.close()
conn.close()
print('Conexión cerrada.')


Conexión cerrada.
